In [ ]:
import scanpy as sc
import squidpy as sq

import numpy as np
from tqdm import tqdm

import torch 
from torch_geometric.data import Data, Dataset
from torch_geometric.utils import k_hop_subgraph, subgraph

In [ ]:
HE22_HUMAN_LUNG_DATA_PATH = '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad'

In [ ]:
params = {
    'data_path'   : HE22_HUMAN_LUNG_DATA_PATH,
    'basis'       : [],
    'node_key'    : ['niche'],
    'graph_id'    : ['window'],
    'radius'      : 30
}

In [ ]:
class spatial_dataset(Dataset):
    """ Dataset that takes anndata, builds neighboorh graphs using squidpy and makes them compatible with PyG
    """
    
    def __init__(self,dict_in):

        self.dict_in = dict_in # Dictionary with parameters to initialize the dataset 
        self.data_path = self.dict_in['data_path']
        self.radius = self.dict_in['radius']
        
        adata = sc.read(self.data_path) # Read anndata with scanpy
        sq.gr.spatial_neighbors(adata, spatial_key='spatial', delaunay=True, coord_type="generic", radius=self.radius)
        self.entire_graph = self._entire_graph(adata)

    def _entire_graph(self, adata):
        """ Function to create a graph with the whole matrix created by Squidpy
        The graph will later be split into subgraphs according to graph_key

        Args:
            adata (anndata): anndata with the info

        Returns:
            graph (torch_geometric.Data): entire graph built from the adjacency matrix
        """

        features = []
        input_edges = np.array([])
        output_edges = np.array([])

        for i in tqdm(range(0,adata.obsp['spatial_connectivities'].shape[0])):
            adjacency_list = adata.obsp['spatial_connectivities'][i,:]
            connections_out = np.where(adjacency_list.A == 1)[1].tolist()
            connections_in = np.array([i]*len(connections_out))
            input_edges = np.concatenate((input_edges, connections_in, connections_out))  # to make to undirected
            output_edges = np.concatenate((output_edges, connections_out, connections_in))
            features.append([adata.X[i,:]]) # Node features: gene expression


        edges_graph = torch.tensor(np.array([input_edges,output_edges]), dtype=torch.long) # 64-bit integer (signed)
        features = torch.tensor(np.array(features), dtype=torch.float)
        features = torch.squeeze(features)

        graph = Data(x=features, edge_index=edges_graph)

        return graph

    def _cell_subgraph(self, cell_idx):
        """ Function to create the subgraph of k-hops with root node the cell of interest (neighbors of the cell)
        The function is called inside _getitem_

        Args:
            cell_idx (int): index of the cell of interest

        Returns:
            subgraph (torch_geometric.Data): subgraph of k-hops with root in the cell queried
        """

        nodes_subgraph, edges_subgraph, _, _ = k_hop_subgraph(node_idx=cell_idx,
                                                              num_hops=self.hops,
                                                              edge_index=self.entire_graph.edge_index,
                                                              relabel_nodes=True,
                                                              )

        subgraph = Data(x=self.entire_graph.x[nodes_subgraph, :], edge_index=edges_subgraph)

        return subgraph

    def __getitem__(self, I):
        """ Function to get the items asked by the dataloader

        Args:
            i (int): query by the dataloader. The dataloader ask for i=0 to i=len(dataset)

        Returns:
            subgraph (torch_geometric.Data): subgraph queried by the dataloader
        """
        
        subgraph = self._cell_subgraph(i)

        return subgraph

    def __len__(self):
        return self.entire_graph.x.shape[0]